In [32]:
import pandas as pd
import numpy as np
import glob
import os

In [ ]:
# read multiple files from one folder and concat it.
# Put all the monthly csv files you want to merge into a single folder, then copy its path

path = '../data/raw'
all_files = glob.glob(os.path.join(path,"*.csv"))

In [ ]:
# check columns name before concat to prevent error.
for f in all_files:
    temp_df = pd.read_csv(f, nrows=0)
    print(f"{os.path.basename(f)}: {temp_df.columns.tolist()}")

202401-divvy-tripdata.csv: ['ride_id', 'rideable_type', 'started_at', 'ended_at', 'start_station_name', 'start_station_id', 'end_station_name', 'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng', 'member_casual']
202402-divvy-tripdata.csv: ['ride_id', 'rideable_type', 'started_at', 'ended_at', 'start_station_name', 'start_station_id', 'end_station_name', 'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng', 'member_casual']
202403-divvy-tripdata.csv: ['ride_id', 'rideable_type', 'started_at', 'ended_at', 'start_station_name', 'start_station_id', 'end_station_name', 'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng', 'member_casual']
202404-divvy-tripdata.csv: ['ride_id', 'rideable_type', 'started_at', 'ended_at', 'start_station_name', 'start_station_id', 'end_station_name', 'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng', 'member_casual']
202405-divvy-tripdata.csv: ['ride_id', 'rideable_type', 'started_at', 'ended_at', 'start_sta

In [ ]:
# concat all file to one dataframe 2024-2026
df_list = [pd.read_csv(f) for f in all_files]
df_final = pd.concat(df_list, ignore_index=True)
print(f'{len(df_final)} rows of data.')

12069836 rows of data.


In [ ]:
# copy df
df = df_final.copy()
print(f'{len(df)} rows of data.')

12069836 rows of data.


In [52]:
df.shape

(12069836, 13)

In [ ]:
# check data type 
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12069836 entries, 0 to 12069835
Data columns (total 13 columns):
 #   Column              Dtype  
---  ------              -----  
 0   ride_id             str    
 1   rideable_type       str    
 2   started_at          str    
 3   ended_at            str    
 4   start_station_name  str    
 5   start_station_id    str    
 6   end_station_name    str    
 7   end_station_id      str    
 8   start_lat           float64
 9   start_lng           float64
 10  end_lat             float64
 11  end_lng             float64
 12  member_casual       str    
dtypes: float64(4), str(9)
memory usage: 2.6 GB


In [54]:
df.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,C1D650626C8C899A,electric_bike,2024-01-12 15:30:27,2024-01-12 15:37:59,Wells St & Elm St,KA1504000135,Kingsbury St & Kinzie St,KA1503000043,41.903267,-87.634737,41.889177,-87.638506,member
1,EECD38BDB25BFCB0,electric_bike,2024-01-08 15:45:46,2024-01-08 15:52:59,Wells St & Elm St,KA1504000135,Kingsbury St & Kinzie St,KA1503000043,41.902937,-87.634440,41.889177,-87.638506,member
2,F4A9CE78061F17F7,electric_bike,2024-01-27 12:27:19,2024-01-27 12:35:19,Wells St & Elm St,KA1504000135,Kingsbury St & Kinzie St,KA1503000043,41.902951,-87.634470,41.889177,-87.638506,member
3,0A0D9E15EE50B171,classic_bike,2024-01-29 16:26:17,2024-01-29 16:56:06,Wells St & Randolph St,TA1305000030,Larrabee St & Webster Ave,13193,41.884295,-87.633963,41.921822,-87.644140,member
4,33FFC9805E3EFF9A,classic_bike,2024-01-31 05:43:23,2024-01-31 06:09:35,Lincoln Ave & Waveland Ave,13253,Kingsbury St & Kinzie St,KA1503000043,41.948797,-87.675278,41.889177,-87.638506,member


In [ ]:
# formatting date columns.
date_cols = ['started_at','ended_at']

for col in date_cols:
    df[col] = pd.to_datetime(df[col],format='mixed')

In [ ]:
# check after format.
print(f'{len(df)} rows of data.')

12069836 rows of data.


In [ ]:
# check missing value (it's enormous, unlikely to be missing at random).
df.isnull().sum()

ride_id                     0
rideable_type               0
started_at                  0
ended_at                    0
start_station_name    2373020
start_station_id      2373020
end_station_name      2473173
end_station_id        2473173
start_lat                   0
start_lng                   0
end_lat                 13375
end_lng                 13375
member_casual               0
month                       0
year                        0
dtype: int64

In [ ]:
# ride_id should be unique as it is an identifier.
df['ride_id'].is_unique

False

In [60]:
# check how many ride_id are duplicated
duplicates = df[df.duplicated('ride_id',keep=False)].sort_values('ride_id')
print(len(duplicates))

422


In [ ]:
# too little so we can delete them.
df = df.drop_duplicates(subset=['ride_id'], keep=False)
if df['ride_id'].is_unique:
    print('ride_id is unique dammit')
else: print('nope')

ride_id is unique dammit


In [ ]:
# duration shoudn't be negative they aren't time travel.
negative_duration = df[df['ended_at'] < df['started_at']]
print(f"Negative duration trips: {len(negative_duration)}")

df = df[df['ended_at'] >= df['started_at']]

Negative duration trips: 256


In [ ]:
# extract new columns.
df['duration'] = df['ended_at'] - df['started_at']
df['duration_min'] = df['duration'] / pd.Timedelta(minutes=1)
df['duration_hour'] = df['duration'] / pd.Timedelta(hours=1)

# exclude riding that have duration less than 1 minute because it unlikely to be real riding.
df = df[df['duration_min'] >= 1]

df.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,month,year,duration,duration_min,duration_hour
0,C1D650626C8C899A,electric_bike,2024-01-12 15:30:27,2024-01-12 15:37:59,Wells St & Elm St,KA1504000135,Kingsbury St & Kinzie St,KA1503000043,41.903267,-87.634737,41.889177,-87.638506,member,1,2024,0 days 00:07:32,7.533333,0.125556
1,EECD38BDB25BFCB0,electric_bike,2024-01-08 15:45:46,2024-01-08 15:52:59,Wells St & Elm St,KA1504000135,Kingsbury St & Kinzie St,KA1503000043,41.902937,-87.634440,41.889177,-87.638506,member,1,2024,0 days 00:07:13,7.216667,0.120278
2,F4A9CE78061F17F7,electric_bike,2024-01-27 12:27:19,2024-01-27 12:35:19,Wells St & Elm St,KA1504000135,Kingsbury St & Kinzie St,KA1503000043,41.902951,-87.634470,41.889177,-87.638506,member,1,2024,0 days 00:08:00,8.000000,0.133333
3,0A0D9E15EE50B171,classic_bike,2024-01-29 16:26:17,2024-01-29 16:56:06,Wells St & Randolph St,TA1305000030,Larrabee St & Webster Ave,13193,41.884295,-87.633963,41.921822,-87.644140,member,1,2024,0 days 00:29:49,29.816667,0.496944
4,33FFC9805E3EFF9A,classic_bike,2024-01-31 05:43:23,2024-01-31 06:09:35,Lincoln Ave & Waveland Ave,13253,Kingsbury St & Kinzie St,KA1503000043,41.948797,-87.675278,41.889177,-87.638506,member,1,2024,0 days 00:26:12,26.200000,0.436667


In [66]:
df.isnull().sum()

ride_id                     0
rideable_type               0
started_at                  0
ended_at                    0
start_station_name    2206194
start_station_id      2206194
end_station_name      2262803
end_station_id        2262803
start_lat                   0
start_lng                   0
end_lat                 13295
end_lng                 13295
member_casual               0
month                       0
year                        0
duration                    0
duration_min                0
duration_hour               0
dtype: int64

In [ ]:
# check for null station because it's 2 million, unlikely to be missing at random or by system failure.
df_null_station_check = df[df['start_station_name'].isnull() | df['end_station_name'].isnull()
                           ].groupby('rideable_type')[['start_station_name','end_station_name']].size()

print('--- n rows of null in both start station and end station ---')
print(len(df[df['start_station_name'].isnull() | df['end_station_name'].isnull()]))

display(df_null_station_check)

In [ ]:
# station fill, assume they were picked and ended up on street because the bycicle has IoT and GPS.
df = df.fillna({
    'start_station_name':'On-street',
    'end_station_name':'On-street',
    'start_station_id':'STREET',
    'end_station_id':'STREET'
})

In [ ]:
# lat, lng mean imputer in case of having station name but no coordinate.
station_coor_map = df.groupby('end_station_name')[['end_lat','end_lng']].mean()
station_coor_map

,end_lat,end_lng
end_station_name,,
2112 W Peterson Ave,41.991178,-87.683593
21st St & Pulaski Rd,41.853506,-87.724802
63rd St Beach,41.780911,-87.576324
900 W Harrison St,41.874754,-87.649807
Aberdeen St & 103rd St,41.707040,-87.650220
...,...,...
Yates Ave & 95th St,41.722370,-87.564780
Yates Blvd & 75th St,41.758768,-87.566440
Yates Blvd & 93rd St,41.726166,-87.566276


In [70]:
df['end_lat'] = df['end_lat'].fillna(df['end_station_name'].map(station_coor_map['end_lat']))
df['end_lng'] = df['end_lng'].fillna(df['end_station_name'].map(station_coor_map['end_lng']))

In [71]:
df.isnull().sum()

ride_id               0
rideable_type         0
started_at            0
ended_at              0
start_station_name    0
start_station_id      0
end_station_name      0
end_station_id        0
start_lat             0
start_lng             0
end_lat               0
end_lng               0
member_casual         0
month                 0
year                  0
duration              0
duration_min          0
duration_hour         0
dtype: int64

In [72]:
%pip install holidays

Note: you may need to restart the kernel to use updated packages.


In [76]:
# more fields extraction 
df['day_of_week'] = df['started_at'].dt.weekday
df['month'] = df['started_at'].dt.month

In [ ]:
# add holidays of chicago.
import holidays

us_il_holidays = holidays.US(state='IL')
df['holiday_name'] = df['started_at'].dt.date.apply(lambda x : us_il_holidays.get(x))

In [78]:
# fill regular days for None
df = df.fillna({'holiday_name':'None'})

In [ ]:
# add season base on astronomy data.
season_data = [
    ('2023-03-20', 'Spring'), ('2023-06-21', 'Summer'), ('2023-09-23', 'Fall'), ('2023-12-21', 'Winter'),
    ('2024-03-19', 'Spring'), ('2024-06-20', 'Summer'), ('2024-09-22', 'Fall'), ('2024-12-21', 'Winter'),
    ('2025-03-20', 'Spring'), ('2025-06-20', 'Summer'), ('2025-09-22', 'Fall'), ('2025-12-21', 'Winter'),
    ('2026-03-20', 'Spring'), ('2026-06-21', 'Summer'), ('2026-09-22', 'Fall'), ('2026-12-21', 'Winter')
]

season_df = pd.DataFrame(season_data, columns=['date','season'])
season_df['date'] = pd.to_datetime(season_df['date'])

season_df = season_df.sort_values('date')

In [81]:
# sorting for merge as of (smart thing to do with this large size of data)
df = df.sort_values('started_at')

df = pd.merge_asof(df, season_df,
                   left_on='started_at',
                   right_on='date',
                   direction='backward')

df.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,...,member_casual,month,year,duration,duration_min,duration_hour,day_of_week,holiday_name,date,season
0,6A9D809ABDF70617,electric_bike,2024-01-01 00:00:39,2024-01-01 00:07:56,On-street,STREET,Clark St & Wrightwood Ave,TA1305000014,41.950000,-87.650000,...,member,1,2024,0 days 00:07:17,7.283333,0.121389,0,New Year's Day,2023-12-21,Winter
1,26B9F6E416A68EBA,electric_bike,2024-01-01 00:00:50,2024-01-01 00:04:20,On-street,STREET,On-street,STREET,41.900000,-87.620000,...,casual,1,2024,0 days 00:03:30,3.500000,0.058333,0,New Year's Day,2023-12-21,Winter
2,30EF016A0DF5ECEE,electric_bike,2024-01-01 00:00:52,2024-01-01 00:04:25,On-street,STREET,On-street,STREET,41.900000,-87.620000,...,casual,1,2024,0 days 00:03:33,3.550000,0.059167,0,New Year's Day,2023-12-21,Winter
3,0D9E85F804C85828,electric_bike,2024-01-01 00:00:53,2024-01-01 00:22:30,Clinton St & Madison St,TA1305000032,On-street,STREET,41.881909,-87.641296,...,member,1,2024,0 days 00:21:37,21.616667,0.360278,0,New Year's Day,2023-12-21,Winter
4,56F5C3ED5178C131,classic_bike,2024-01-01 00:01:01,2024-01-01 00:24:12,LaSalle St & Illinois St,13430,Indiana Ave & Roosevelt Rd,SL-005,41.890762,-87.631697,...,member,1,2024,0 days 00:23:11,23.183333,0.386389,0,New Year's Day,2023-12-21,Winter


In [ ]:
# clear unnesescery column after merge.
df = df.drop(columns=['date'])
df.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,...,end_lng,member_casual,month,year,duration,duration_min,duration_hour,day_of_week,holiday_name,season
0,6A9D809ABDF70617,electric_bike,2024-01-01 00:00:39,2024-01-01 00:07:56,On-street,STREET,Clark St & Wrightwood Ave,TA1305000014,41.950000,-87.650000,...,-87.643118,member,1,2024,0 days 00:07:17,7.283333,0.121389,0,New Year's Day,Winter
1,26B9F6E416A68EBA,electric_bike,2024-01-01 00:00:50,2024-01-01 00:04:20,On-street,STREET,On-street,STREET,41.900000,-87.620000,...,-87.620000,casual,1,2024,0 days 00:03:30,3.500000,0.058333,0,New Year's Day,Winter
2,30EF016A0DF5ECEE,electric_bike,2024-01-01 00:00:52,2024-01-01 00:04:25,On-street,STREET,On-street,STREET,41.900000,-87.620000,...,-87.620000,casual,1,2024,0 days 00:03:33,3.550000,0.059167,0,New Year's Day,Winter
3,0D9E85F804C85828,electric_bike,2024-01-01 00:00:53,2024-01-01 00:22:30,Clinton St & Madison St,TA1305000032,On-street,STREET,41.881909,-87.641296,...,-87.620000,member,1,2024,0 days 00:21:37,21.616667,0.360278,0,New Year's Day,Winter
4,56F5C3ED5178C131,classic_bike,2024-01-01 00:01:01,2024-01-01 00:24:12,LaSalle St & Illinois St,13430,Indiana Ave & Roosevelt Rd,SL-005,41.890762,-87.631697,...,-87.623041,member,1,2024,0 days 00:23:11,23.183333,0.386389,0,New Year's Day,Winter


In [ ]:
# check data type again.
df.dtypes

ride_id                           str
rideable_type                     str
started_at             datetime64[us]
ended_at               datetime64[us]
start_station_name                str
start_station_id                  str
end_station_name                  str
end_station_id                    str
start_lat                     float64
start_lng                     float64
end_lat                       float64
end_lng                       float64
member_casual                     str
month                           int32
year                            int32
duration              timedelta64[us]
duration_min                  float64
duration_hour                 float64
day_of_week                     int32
holiday_name                      str
season                            str
dtype: object

In [ ]:
# check columns.
df.columns

Index(['ride_id', 'rideable_type', 'started_at', 'ended_at',
       'start_station_name', 'start_station_id', 'end_station_name',
       'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng',
       'member_casual', 'month', 'year', 'duration', 'duration_min',
       'duration_hour', 'day_of_week', 'holiday_name', 'season'],
      dtype='str')

In [ ]:
# re-order columns.
df_reordered = df[['ride_id', 'rideable_type', 'started_at', 'ended_at',
        'day_of_week', 'holiday_name', 'season', 'month', 'year',
       'start_station_name', 'start_station_id', 'end_station_name',
       'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng',
       'member_casual', 'duration', 'duration_min', 'duration_hour']]

df_master = df_reordered.copy()

In [ ]:
# check final data.
df_master

,ride_id,rideable_type,started_at,ended_at,day_of_week,holiday_name,season,month,year,start_station_name,...,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,duration,duration_min,duration_hour
0,6A9D809ABDF70617,electric_bike,2024-01-01 00:00:39.000,2024-01-01 00:07:56.000,0,New Year's Day,Winter,1,2024,On-street,...,Clark St & Wrightwood Ave,TA1305000014,41.950000,-87.650000,41.929546,-87.643118,member,0 days 00:07:17,7.283333,0.121389
1,26B9F6E416A68EBA,electric_bike,2024-01-01 00:00:50.000,2024-01-01 00:04:20.000,0,New Year's Day,Winter,1,2024,On-street,...,On-street,STREET,41.900000,-87.620000,41.900000,-87.620000,casual,0 days 00:03:30,3.500000,0.058333
2,30EF016A0DF5ECEE,electric_bike,2024-01-01 00:00:52.000,2024-01-01 00:04:25.000,0,New Year's Day,Winter,1,2024,On-street,...,On-street,STREET,41.900000,-87.620000,41.900000,-87.620000,casual,0 days 00:03:33,3.550000,0.059167
3,0D9E85F804C85828,electric_bike,2024-01-01 00:00:53.000,2024-01-01 00:22:30.000,0,New Year's Day,Winter,1,2024,Clinton St & Madison St,...,On-street,STREET,41.881909,-87.641296,41.890000,-87.620000,member,0 days 00:21:37,21.616667,0.360278
4,56F5C3ED5178C131,classic_bike,2024-01-01 00:01:01.000,2024-01-01 00:24:12.000,0,New Year's Day,Winter,1,2024,LaSalle St & Illinois St,...,Indiana Ave & Roosevelt Rd,SL-005,41.890762,-87.631697,41.867888,-87.623041,member,0 days 00:23:11,23.183333,0.386389
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11774324,4C45D094537F5887,electric_bike,2026-03-31 23:47:48.869,2026-03-31 23:54:37.125,1,None,Spring,3,2026,Clark St & Newport St,...,Clark St & Wrightwood Ave,CHI00287,41.944540,-87.654678,41.929546,-87.643118,casual,0 days 00:06:48.256000,6.804267,0.113404
11774325,67860E0EAEE7390B,electric_bike,2026-03-31 23:48:07.066,2026-03-31 23:54:42.804,1,None,Spring,3,2026,Clark St & Newport St,...,Clark St & Wrightwood Ave,CHI00287,41.944540,-87.654678,41.929546,-87.643118,member,0 days 00:06:35.738000,6.595633,0.109927
11774326,F3465D96B1E74615,classic_bike,2026-03-31 23:48:10.515,2026-03-31 23:51:24.482,1,None,Spring,3,2026,Lake Park Ave & 53rd St,...,Kimbark Ave & 53rd St,CHI00490,41.799494,-87.586450,41.799455,-87.594778,member,0 days 00:03:13.967000,3.232783,0.053880
11774327,382BC0409C3C7147,classic_bike,2026-03-31 23:55:43.748,2026-03-31 23:59:47.821,1,None,Spring,3,2026,California Ave & Milwaukee Ave,...,Rockwell Ave & Logan Blvd,CHI02130,41.922695,-87.697153,41.928110,-87.692680,member,0 days 00:04:04.073000,4.067883,0.067798


In [ ]:
# save to .parquet file for further analysis (.csv is too big so she tell you not to worry about).

# import time

# start = time.time()
# df_master.to_parquet('../data/processed/cyclistic_cleaned_data.parquet')
# end = time.time()

# parquet_time_read = end - start
# print('time taken in seconds: ', parquet_time_read)

time taken in seconds:  22.299673318862915


**Summarize what have done in this file.**

1. Read multiple files and merge it to one file.

2. Checks anonaly in the data to make sure about it's quality.

3. Clean the data.

4. Adding useful features or columns for Analysis.

5. Export to Parquet file.